In [1]:
import pandas as pd
import numpy as np
import time
import json
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [3]:
from ufal.udpipe import Model, Pipeline
import ufal.udpipe

In [4]:
# Configuration
MODEL_NAME = "UDPipe"
SAMPLE_TYPES = ["glosses", "medieval_charters"]
TASKS = ["lemmatization", "pos_tagging"]

# Notebook path
notebook_path = os.path.abspath("04-udpipe.ipynb")

In [5]:
# Results storage
results = {
    "model_name": MODEL_NAME,
    "processing_times": {},
    "accuracy": {},
    "precision": {},
    "recall": {},
    "f1_score": {}
}

In [6]:
def word_joiner(gold_df):
    # Extract text to reconstruct from words

    sample_texts = []
    for sample_id in gold_df['sample_id'].unique():
        words = gold_df[gold_df['sample_id'] == sample_id]['word'].tolist()
        text = ' '.join(words)
        sample_texts.append((sample_id, text))

    return sample_texts

In [15]:
def udpipe_processor(sample_texts, sample_type, pipeline):
    processed_results = []
    for sample_id, text in sample_texts:

        processed = pipeline.process(text)
        sentences = processed.strip().split('\n\n')

        for sent in sentences:
            lines = sent.split("\n")
            for idx, line in enumerate(lines):
                if line.startswith('#') or not line.strip():
                    continue

                fields = line.split('\t')
                if len(fields) != 10:
                    continue

                form = fields[1]
                lemma = fields[2]
                upos = fields[3]

                if sample_type == "glosses":
                    word_id = f"http://gams.uni-graz.at/o:glossvibe.bvi#{sample_id}.{idx - 1}"
                else:
                    word_id = str(idx)

                processed_results.append({
                    "sample_id": sample_id,
                    "word_id": word_id,
                    "word": form,
                    "lemma": lemma,
                    "pos": upos
                })

    return processed_results


In [8]:
def analyse(merged_df, sample_type):
    # Convert the lemma columns to string type to ensure consistent comparison
    merged_df['lemma_gold'] = merged_df['lemma_gold'].astype(str)
    merged_df['lemma_pred'] = merged_df['lemma_pred'].astype(str)

    # Evaluate lemmatization
    lemma_accuracy = accuracy_score(merged_df['lemma_gold'], merged_df['lemma_pred'])

    # Create a binary array where True means the prediction matches the gold standard
    matches = merged_df['lemma_gold'] == merged_df['lemma_pred']

    # Fix the precision_recall_fscore_support call
    lemma_precision, lemma_recall, lemma_f1, _ = precision_recall_fscore_support(
        matches,
        [True] * len(merged_df),
        average='binary'
    )

    # Convert the lemma columns to string type to ensure consistent comparison
    merged_df['pos_gold'] = merged_df['pos_gold'].astype(str)
    merged_df['pos_pred'] = merged_df['pos_pred'].astype(str)

    # Evaluate lemmatization
    pos_accuracy = accuracy_score(merged_df['pos_gold'], merged_df['pos_pred'])

    # Create a binary array where True means the prediction matches the gold standard
    matches = merged_df['pos_gold'] == merged_df['pos_pred']

    # Fix the precision_recall_fscore_support call
    pos_precision, pos_recall, pos_f1, _ = precision_recall_fscore_support(
        matches,
        [True] * len(merged_df),
        average='binary'
    )

    results["accuracy"][f"{sample_type}_lemma"] = lemma_accuracy
    results["precision"][f"{sample_type}_lemma"] = lemma_precision
    results["recall"][f"{sample_type}_lemma"] = lemma_recall
    results["f1_score"][f"{sample_type}_lemma"] = lemma_f1

    results["accuracy"][f"{sample_type}_pos"] = pos_accuracy
    results["precision"][f"{sample_type}_pos"] = pos_precision
    results["recall"][f"{sample_type}_pos"] = pos_recall
    results["f1_score"][f"{sample_type}_pos"] = pos_f1

    merged_df.to_csv(f"../results/{MODEL_NAME}_{sample_type}_detailed.csv", index=False)

    print(f"Completed {sample_type}. Processing time: {processing_time:.2f}s")
    print(f"Lemmatization accuracy: {lemma_accuracy:.4f}")
    print(f"POS tagging accuracy: {pos_accuracy:.4f}")
    print("-" * 50)


In [10]:
# Load the UDPipe model
model_path = "/Users/Thea/Desktop/LatinNLPTools/scripts/latin-ittb-ud-2.5-191206.udpipe"
model = Model.load(model_path)
if not model:
    raise Exception("Model not loaded!")

In [11]:
# Create a processing pipeline
pipeline = Pipeline(model, "tokenize", Pipeline.DEFAULT, Pipeline.DEFAULT, "conllu")

In [16]:
for sample_type in SAMPLE_TYPES:
    print(f"Processing {sample_type}...")

    # Load gold standard data
    gold_file = os.path.join(os.path.dirname(notebook_path), f"../data/gold_standard/gs_{sample_type}.csv")

    gold_df = pd.read_csv(gold_file)

    # Extract text to reconstruct from words
    sample_texts = word_joiner(gold_df)

    # Process samples and measure time
    start_time = time.time()

    processed_results = udpipe_processor(sample_texts, sample_type, pipeline)

    processing_time = time.time() - start_time
    results["processing_times"][sample_type] = processing_time

    print(f"Data processes with {MODEL_NAME} in {processing_time} seconds.")

    # Merge gold_df with processed_results
    pred_df = pd.DataFrame(processed_results)

    # Add a token index per sample in both gold and pred dataframes
    gold_df['token_idx'] = gold_df.groupby('sample_id').cumcount()
    pred_df['token_idx'] = pred_df.groupby('sample_id').cumcount()

    # Merge on sample_id and token index
    merged_df = pd.merge(gold_df, pred_df, on=['sample_id', 'token_idx'], suffixes=('_gold', '_pred'))

    # Check mismatches
    merged_df['word_match'] = merged_df['word_gold'] == merged_df['word_pred']
    print("Token mismatches:")
    print(merged_df[~merged_df['word_match']].head())

    merged_df['word_match'] = merged_df['word_gold'] == merged_df['word_pred']
    print("Mismatched word rows:")
    mismatched_df = merged_df[~merged_df['word_match']].head()
    print(len(mismatched_df))

    print(f"Running analysis on {sample_type}...")
    analyse(merged_df, sample_type)




Processing glosses...
Data processes with UDPipe in 0.18763303756713867 seconds.
Token mismatches:
      sample_id                                       word_id_gold word_gold  \
575   BVi.04a10  http://gams.uni-graz.at/o:glossvibe.bvi#BVi.04...       ·l·   
576   BVi.04a10  http://gams.uni-graz.at/o:glossvibe.bvi#BVi.04...    satius   
596  Ang.58a13b  http://gams.uni-graz.at/o:glossvibe.bvi#Ang.58...        s.   
597  Ang.58a13b  http://gams.uni-graz.at/o:glossvibe.bvi#Ang.58...    contra   
598  Ang.58a13b  http://gams.uni-graz.at/o:glossvibe.bvi#Ang.58...   dixerit   

           lemma_gold pos_gold  token_idx  \
575  (Roman numerals)      NUM          0   
576            satius     NOUN          1   
596             idest    CCONJ          0   
597            contra      ADP          1   
598              dico     VERB          2   

                                          word_id_pred word_pred lemma_pred  \
575  http://gams.uni-graz.at/o:glossvibe.bvi#BVi.04...        ·l      

In [17]:
# Save summary results
with open(f"../results/{MODEL_NAME}_summary.json", "w") as f:
    json.dump(results, f, indent=2)